In [22]:
import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine
from datetime import date

#### Conexion con las bases de datos

In [23]:
with open("../config.yml", "r", encoding='utf-8') as f:
    config = yaml.safe_load(f)

    config_co = config['SOURCE_DB']
    config_etl = config['ETL_PRO']

url_co = f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:{config_co['port']}/{config_co['dbname']}"
url_etl = f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:{config_etl['port']}/{config_etl['dbname']}"


engine_origen = create_engine(url_co)
engine_destino = create_engine(url_etl)

In [24]:
dim_cliente = pd.read_sql_table('dim_cliente', engine_destino, schema='data_mart_entregas')
dim_sede = pd.read_sql_table('dim_sede', engine_destino, schema='data_mart_entregas')
dim_mensajero = pd.read_sql_table('dim_mensajero', engine_destino, schema='data_mart_entregas')
dim_tiempo = pd.read_sql_table('dim_tiempo', engine_destino, schema='data_mart_entregas')
dim_hora = pd.read_sql_table('dim_hora', engine_destino, schema='data_mart_entregas')

In [25]:
dim_cliente.head()

,id_cliente,nombre_cliente,tipo_cliente,email,telefono,sector
0,1,Cliente 2,1,algo.com,327-00000,S
1,2,Cliente 1,1,algo.com,327-00000,industrial
2,6,CLINICA DEPORTIVA DEL SUR,1,algo.com,327-00000,salud
3,19,HOSPITAL ORTOPEDICO DE COLOMBIA,1,algo.com,327-00000,salud
4,8,CLINICA NEFROLOGOS DE CALI,1,algo.com,327-00000,salud


In [26]:
dim_sede.head()

,id_sede,nombre_sede,ciudad,departamento,direccion
0,10,FARALLONES /123,CALI,VALLE DEL CAUCA,Los angeles distrito Latino
1,11,REMEDIOZ/ 123,CALI,VALLE DEL CAUCA,Los angeles distrito Latino
2,13,DIME / LOS ROJOS,CALI,VALLE DEL CAUCA,Los angeles distrito Latino
3,14,DESPACHOS / LOS ROJOS,CALI,VALLE DEL CAUCA,Los angeles distrito Latino
4,23,POPAYAN BODEGA 28 / A,POPAYAN,CAUCA,Los angeles distrito Latino


In [27]:
dim_mensajero.head()

,id_mensajero,activo
0,1,True
1,42,True
2,48,True
3,41,True
4,13,True


In [28]:
dim_tiempo.head()

,fecha,id_tiempo,año,mes,dia,dia_semana,fin_de_semana
0,2023-09-19,1,2023,9,19,1,0
1,2023-09-20,2,2023,9,20,2,0
2,2023-09-21,3,2023,9,21,3,0
3,2023-09-22,4,2023,9,22,4,0
4,2023-09-23,5,2023,9,23,5,1


In [29]:
dim_hora.head()

,id_hora,hora
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4


In [30]:
servicio = pd.read_sql_table('mensajeria_servicio', engine_origen)
usuario = pd.read_sql_table('clientes_usuarioaquitoy', engine_origen)
estados = pd.read_sql_table('mensajeria_estadosservicio', engine_origen)

In [31]:
df = pd.merge(servicio, usuario[['id', 'sede_id']], left_on='usuario_id', right_on='id', how='left')

In [32]:
df['datetime_solicitud'] = pd.to_datetime(
    df['fecha_solicitud'].astype(str) + ' ' + df['hora_solicitud'].astype(str),
    format='mixed'
)

In [33]:
estados['fecha_str'] = pd.to_datetime(estados['fecha']).dt.strftime('%Y-%m-%d')
estados['hora_str'] = estados['hora'].astype(str).str.split('.').str[0]
estados['datetime'] = pd.to_datetime(
    estados['fecha_str'] + ' ' + estados['hora_str'],
    format='%Y-%m-%d %H:%M:%S'
)

duracion = estados.groupby('servicio_id')['datetime'].agg(
    inicio='min', fin='max'
).reset_index()

duracion['duracion_servicio'] = (
    (duracion['fin'] - duracion['inicio'])
    .dt.total_seconds()
    .div(60)
    .fillna(0)
    .round()
    .astype(int)
)

In [34]:
df = pd.merge(df, duracion[['servicio_id', 'duracion_servicio']], left_on='id_x', right_on='servicio_id', how='left')

In [35]:
dim_tiempo['fecha'] = pd.to_datetime(dim_tiempo['fecha']).dt.tz_localize(None)
df['fecha_solicitud_norm'] = pd.to_datetime(df['fecha_solicitud']).dt.normalize()
df = df.merge(dim_tiempo[['id_tiempo', 'fecha']], left_on='fecha_solicitud_norm', right_on='fecha', how='left')

In [36]:
df['mensajero_id'] = df['mensajero_id'].fillna(0).astype(int)

In [37]:
hecho_servicio = pd.DataFrame()

In [38]:
hecho_servicio['id_hecho_servicio'] = df['id_x']
hecho_servicio['id_cliente']        = df['cliente_id']
hecho_servicio['id_sede']           = df['sede_id']
hecho_servicio['id_tiempo']         = df['id_tiempo'].fillna(1).astype(int)
hecho_servicio['id_hora']           = df['datetime_solicitud'].dt.hour
hecho_servicio['id_mensajero']      = df['mensajero_id']
hecho_servicio['cod_servicio']      = df['id_x']
hecho_servicio['duracion_servicio'] = df['duracion_servicio'].fillna(0).astype(int)
hecho_servicio['total_servicios']   = 1

In [39]:
hecho_servicio = hecho_servicio[df['es_prueba'] == False].reset_index(drop=True)

In [40]:
columnas_finales = [
    'id_hecho_servicio',
    'id_cliente',
    'id_sede',
    'id_tiempo',
    'id_hora',
    'id_mensajero',
    'cod_servicio',
    'duracion_servicio',
    'total_servicios',
]

hecho_servicio = hecho_servicio[columnas_finales]

In [41]:
hecho_servicio.head()

,id_hecho_servicio,id_cliente,id_sede,id_tiempo,id_hora,id_mensajero,cod_servicio,duracion_servicio,total_servicios
0,34,5,6,38,9,0,34,0,1
1,36,5,5,40,19,0,36,0,1
2,41,5,2,50,9,0,41,0,1
3,42,5,2,50,9,0,42,0,1
4,43,5,8,50,10,0,43,0,1


In [42]:
hecho_servicio.to_sql('hecho_servicio', con=engine_destino, schema='data_mart_entregas', if_exists='replace', index=False)
len(hecho_servicio)

28328